# Classification Modelling Pipeline

This notebook walks you step-by-step through building, evaluating, and preparing a **binary classification model** (XGBoost) for deployment.

**What this notebook covers:**
1. Data loading & initial inspection
1a. **15% holdout split** — true out-of-sample set (before any preprocessing)
2. Dropping ID / useless columns
3. Null analysis & column removal by threshold
4. Class-wise variation analysis (to detect low-information columns)
5. Identify categorical vs numerical columns
6. Numerical handling — Supervised Binning
7. Categorical handling — One-Hot Encoding
8. External data — Load, merge & full pipeline (drop/keep, nulls, variation, bin, OHE)
9. Save all transformation artefacts for deployment
10. Stratified train / test split (on dev set)
11. XGBoost training with GridSearchCV
12. Feature importance — smart auto-flag & selection
13. Evaluation metrics — F1, AUC, Gini, Accuracy, Balanced Accuracy (dev test set)
14. Save final model & artefacts
15. Deployment reference
16. **Holdout evaluation** — score raw holdout through deployment pipeline & compare

---

## 0. Install & Import Dependencies

Run the cell below to make sure every package we need is available.

In [ ]:
# ── Install if missing ──────────────────────────────────────────────
# !pip install -r requirements.txt

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from scipy.stats import ks_2samp, chi2_contingency
from xgboost import XGBClassifier

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Directory where we persist artefacts for deployment.
# Uses home directory to avoid Windows CWD / device errors.
ARTEFACT_DIR = str(Path.home() / "ml_artefacts")
os.makedirs(ARTEFACT_DIR, exist_ok=True)

print(f"Artefact directory: {ARTEFACT_DIR}")
print("All imports successful.")

### Helper Functions

These are reused by both the main pipeline and the external data pipeline.

In [ ]:
def cramers_v(col, target):
    """Cramer's V between a categorical column and binary target."""
    ct = pd.crosstab(col, target)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0


def null_analysis(dataframe, columns, threshold_pct, target_col, title=""):
    """Show null % bar chart and return columns above threshold."""
    null_pct = (dataframe[columns].isnull().sum() / len(dataframe) * 100).sort_values(ascending=False)
    null_pct_df = null_pct.reset_index()
    null_pct_df.columns = ["column", "null_pct"]

    cols_with_nulls = null_pct_df[null_pct_df["null_pct"] > 0]
    if len(cols_with_nulls) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(cols_with_nulls) * 0.4)))
        sns.barplot(data=cols_with_nulls, x="null_pct", y="column", ax=ax, palette="Reds_r")
        ax.axvline(threshold_pct, color="black", ls="--", label=f"Threshold {threshold_pct}%")
        ax.set_xlabel("Null %")
        ax.set_title(f"Null Percentage by Column {title}")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print(f"No columns have null values {title}.")

    cols_to_drop = null_pct[null_pct > threshold_pct].index.tolist()
    if target_col in cols_to_drop:
        cols_to_drop.remove(target_col)
    return cols_to_drop


def variation_analysis(dataframe, columns, target_col, threshold, title=""):
    """Compute class-wise variation scores and return flagged columns."""
    variation_scores = {}
    for col in columns:
        if dataframe[col].nunique() < 2:
            variation_scores[col] = 0.0
            continue
        if dataframe[col].dtype in ["object", "category"] or dataframe[col].nunique() <= 10:
            variation_scores[col] = cramers_v(dataframe[col].fillna("__NULL__"), dataframe[target_col])
        else:
            classes = dataframe[target_col].unique()
            grp0 = dataframe.loc[dataframe[target_col] == classes[0], col].dropna()
            grp1 = dataframe.loc[dataframe[target_col] == classes[1], col].dropna()
            if len(grp0) == 0 or len(grp1) == 0:
                variation_scores[col] = 0.0
            else:
                ks_stat, _ = ks_2samp(grp0, grp1)
                variation_scores[col] = ks_stat

    var_df = (
        pd.DataFrame.from_dict(variation_scores, orient="index", columns=["variation_score"])
        .sort_values("variation_score", ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, max(4, len(var_df) * 0.35)))
    colors = ["#d62728" if v < threshold else "#2ca02c" for v in var_df["variation_score"]]
    sns.barplot(x=var_df["variation_score"], y=var_df.index, palette=colors, ax=ax)
    ax.axvline(threshold, ls="--", color="black", label=f"Threshold={threshold}")
    ax.set_title(f"Class-wise Variation Score {title}")
    ax.set_xlabel("Variation Score (higher = more discriminative)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    flagged = var_df[var_df["variation_score"] < threshold].index.tolist()
    return flagged, var_df


def identify_col_types(dataframe, feature_cols):
    """Split feature columns into categorical and numerical."""
    cat = [c for c in feature_cols if dataframe[c].dtype in ["object", "category"] or dataframe[c].nunique() <= 10]
    num = [c for c in feature_cols if c not in cat]
    return cat, num


def bin_columns(dataframe, num_cols, n_bins, strategy, custom_bins, bin_store):
    """Bin numerical columns and return updated dataframe and bin_store."""
    for col in num_cols:
        series = dataframe[col].dropna()
        print(f"\n{'='*60}")
        print(f"Column: {col}")
        print(f"  min={series.min():.4f}  max={series.max():.4f}  "
              f"mean={series.mean():.4f}  median={series.median():.4f}  "
              f"std={series.std():.4f}  nulls={dataframe[col].isnull().sum()}")

        if col in custom_bins:
            edges = custom_bins[col]
            dataframe[col + "_bin"] = pd.cut(dataframe[col], bins=edges, labels=False, include_lowest=True)
            bin_store[col] = {"type": "custom", "edges": edges}
            print(f"  -> Custom bins applied: {edges}")
        else:
            if strategy == "quantile":
                quantiles = np.linspace(0, 1, n_bins + 1)
                edges = list(series.quantile(quantiles).values)
                print(f"  -> Quantile bin edges: {[round(e, 4) for e in edges]}")
            elif strategy == "uniform":
                edges = list(np.linspace(series.min(), series.max(), n_bins + 1))
                print(f"  -> Uniform bin edges:  {[round(e, 4) for e in edges]}")
            else:
                print(f"  -> KMeans binning with {n_bins} bins")

            binner = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy=strategy, subsample=None)
            valid_mask = dataframe[col].notna()
            dataframe.loc[valid_mask, col + "_bin"] = binner.fit_transform(
                dataframe.loc[valid_mask, [col]]
            ).ravel()
            actual_edges = binner.bin_edges_[0].tolist()
            bin_store[col] = {"type": strategy, "edges": actual_edges, "n_bins": n_bins}
            print(f"  -> Actual bin edges saved: {[round(e, 4) for e in actual_edges]}")

        print(f"  -> Value counts after binning:")
        print(dataframe[col + "_bin"].value_counts().sort_index())

    dataframe.drop(columns=num_cols, inplace=True)
    return dataframe, bin_store


def ohe_columns(dataframe, target_col, drop_first, max_categories, rare_map_store, ohe_store=None):
    """One-hot encode all feature columns. Returns encoded df, encoder, rare_mappings."""
    encode_cols = [c for c in dataframe.columns if c != target_col]

    if max_categories is not None:
        for col in encode_cols:
            counts = dataframe[col].value_counts()
            rare_cats = counts[counts < max_categories].index.tolist()
            if rare_cats:
                dataframe[col] = dataframe[col].apply(lambda x: "__rare__" if x in rare_cats else x)
                rare_map_store[col] = rare_cats
                print(f"{col}: grouped {len(rare_cats)} rare categories into '__rare__'")

    for col in encode_cols:
        if dataframe[col].isnull().any():
            dataframe[col] = dataframe[col].fillna("__NULL__")

    dataframe[encode_cols] = dataframe[encode_cols].astype(str)
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore",
                        drop="first" if drop_first else None)
    encoded = ohe.fit_transform(dataframe[encode_cols])
    encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(encode_cols),
                              index=dataframe.index)
    result = pd.concat([encoded_df, dataframe[[target_col]]], axis=1)
    return result, encode_cols, ohe, rare_map_store


print("Helper functions defined.")

---
## 1. Data Loading

**Instructions:**
1. Set `DATA_PATH` to the path of your CSV (or change the reader for other formats).
2. Set `TARGET_COL` to the name of your binary target column.
3. Run the cell — it will print shape, dtypes, and the first few rows so you can verify.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
DATA_PATH  = "connected_dataset_dpd0_20260207_141810.csv"        # <-- change this to your file path
TARGET_COL = "target"          # <-- change this to your target column name
# ────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
df.rename(columns={"target_dpd0": TARGET_COL}, inplace=True)  # Standardize target column name for pipeline
print(f"\nTarget distribution:\n{df[TARGET_COL].value_counts(normalize=True)}")
print(f"\nColumn dtypes:\n{df.dtypes}")
df.head()

---
## 1a. Holdout Split — 15% True Out-of-Sample

**Before any preprocessing**, we set aside **15%** of the data as a completely untouched holdout set.  
This holdout will:
- **Never** be seen during preprocessing (bin edges, OHE fitting, null analysis, variation analysis).  
- Be scored at the very end through the saved deployment artefacts, exactly like production data.  
- Validate that the model generalises and that the deployment pipeline works correctly.

The remaining **85%** goes through the full preprocessing → train/test split → training pipeline.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
HOLDOUT_PCT    = 0.15   # fraction to reserve as true out-of-sample
HOLDOUT_SEED   = 99     # separate seed so it's independent of other splits
# ────────────────────────────────────────────────────────────────────

df_dev, df_holdout = train_test_split(
    df,
    test_size=HOLDOUT_PCT,
    random_state=HOLDOUT_SEED,
    stratify=df[TARGET_COL],
)

# Save the raw holdout immediately (completely untouched)
holdout_raw = df_holdout.copy()

# Continue the pipeline with the dev set only
df = df_dev.reset_index(drop=True)

print(f"Full dataset:   {len(df_dev) + len(df_holdout)} rows")
print(f"Dev set (85%):  {len(df)} rows  — goes through preprocessing + train/test")
print(f"Holdout (15%):  {len(holdout_raw)} rows  — completely raw, scored at the end")
print(f"\nDev target distribution:\n{df[TARGET_COL].value_counts(normalize=True)}")
print(f"\nHoldout target distribution:\n{holdout_raw[TARGET_COL].value_counts(normalize=True)}")

---
## 2. Drop ID / Useless Columns

**Instructions:**  
Look at the columns printed above. Identify any **ID columns**, **row-index columns**, or columns that are clearly **not useful** for prediction (e.g. names, unique identifiers, free-text keys).

Fill in the list below with those column names, then choose:
- `MODE = "drop"` — drop only the columns you list  
- `MODE = "keep"` — keep *only* the columns you list (plus the target)

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
MODE = "keep"  # "drop" or "keep"

COLUMNS_LIST = [
    "target",
    "l_principal",
    "l_partner_disburcement_amount",
    "l_discount",
    "l_term",
    "cl_cupo",
    "cl_cupo_disponible",
    "cedula",
    "platam_score",
    "hybrid_score",
    "platam_rating",
    "hybrid_rating",
    "categoria_madurez",
    "edad",
    "ingresos_smlv",
    "nivel_ingresos_encoded",
    "cuota_mensual",
    "ratio_cuota_ingreso",
    "creditos_vigentes",
    "creditos_mora",
    "hist_neg_12m",
    "departamento",
    "experian_score",
    "total_debt",
    "queries_6m",
    "queries_12m",
    "active_credits",
    "closed_credits",
    "total_entities",
    "negative_entities",
    "genero",
    "edad_promedio",
    "num_accounts",
    "total_account_balance",
    "total_arrears_balance",
    "num_accounts_in_arrears",
    "clr_type",
    "clr_doc_type",
    "clr_bus_relation",
    "clr_city",
    "clr_bus_type",
    "clr_bus_num_locations",
    "clr_bus_num_employees",
    "clr_bus_seniority",
    "clr_bus_monthly_purchases",
    "clr_bus_current_purchases",
    "clr_bus_monthly_income",
    "clr_bus_monthly_expenses",
    "clr_declares_rent",
    "clr_has_rent",
    "clr_rent",
    "clr_hcpn_status",
    "clr_credit_study_score",
    "clr_credit_study_result",
    "clr_credit_study_loc",
    "clr_requested_loc",
    "clr_risk_profile",
    "clr_sr_opinion_relationship_duration",
    "clr_agent_recommendation",
    "clr_agent_loc",
]
# ────────────────────────────────────────────────────────────────────

if MODE == "drop":
    df.drop(columns=[c for c in COLUMNS_LIST if c in df.columns], inplace=True)
    print(f"Dropped {COLUMNS_LIST}")
elif MODE == "keep":
    keep = list(set(COLUMNS_LIST + [TARGET_COL]))
    df = df[[c for c in keep if c in df.columns]]
    print(f"Kept only {len(keep)} columns")

print(f"Remaining shape: {df.shape}")
df.head()

---
## 3. Null Analysis & Column Removal

We will:
1. Show the **percentage of nulls** per column.
2. Drop every column whose null % exceeds the threshold you set.

**Instructions:**  
Set `NULL_THRESHOLD_PCT` — any column with a higher null percentage will be removed.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
NULL_THRESHOLD_PCT = 40  # drop columns with more than this % nulls
# ────────────────────────────────────────────────────────────────────

feature_cols = [c for c in df.columns if c != TARGET_COL]
cols_to_drop = null_analysis(df, feature_cols, NULL_THRESHOLD_PCT, TARGET_COL, title="(Main Data)")

df.drop(columns=cols_to_drop, inplace=True)
print(f"\nDropped {len(cols_to_drop)} columns above {NULL_THRESHOLD_PCT}% nulls: {cols_to_drop}")
print(f"Remaining shape: {df.shape}")

---
## 4. Class-wise Variation Analysis

For each feature we compare its distribution across the two target classes.  
Columns that look **almost identical** in both classes carry little predictive power and can be dropped.

**How we measure it:**
- **Numerical columns** → Kolmogorov-Smirnov test. Higher KS stat = distributions differ more.
- **Categorical columns** → Cramer's V. Higher V = stronger association with target.

**Instructions:**  
Set `VARIATION_THRESHOLD` — columns below this threshold will be flagged.  
Review the flagged columns and decide which to drop.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
VARIATION_THRESHOLD = 0.02  # columns with variation score below this get flagged
# ────────────────────────────────────────────────────────────────────

feature_cols = [c for c in df.columns if c != TARGET_COL]
flagged, var_df = variation_analysis(df, feature_cols, TARGET_COL, VARIATION_THRESHOLD, title="(Main Data)")
print(f"\nFlagged columns (score < {VARIATION_THRESHOLD}): {flagged}")

**Instructions:**  
Review the flagged columns above. Add any you want to drop to the list below.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
DROP_LOW_VARIATION = [
    "clr_bus_current_purchases",
    "clr_requested_loc",
    "clr_rent",
]
# ────────────────────────────────────────────────────────────────────

df.drop(columns=[c for c in DROP_LOW_VARIATION if c in df.columns], inplace=True)
print(f"Dropped {len(DROP_LOW_VARIATION)} low-variation columns.")
print(f"Remaining shape: {df.shape}")

---
## 5. Identify Categorical vs Numerical Columns

The notebook auto-detects column types.  
Review the lists and move columns between them if the auto-detection was wrong.

In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COL]
cat_cols, num_cols = identify_col_types(df, feature_cols)

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns  ({len(num_cols)}): {num_cols}")

**Instructions:**  
If any column was misclassified, move it manually in the cell below.

In [ ]:
# ── USER INPUT (optional) ──────────────────────────────────────────
# Move columns between lists if the auto-detection was wrong:
# cat_cols.append("some_numeric_col_that_is_actually_categorical")
# num_cols.remove("some_numeric_col_that_is_actually_categorical")
# ────────────────────────────────────────────────────────────────────

print(f"Final categorical: {cat_cols}")
print(f"Final numerical:   {num_cols}")

---
## 6. Numerical Columns — Binning

For each numerical column we will:
1. Show its distribution and the **binning conditions** before applying.
2. Bin it into discrete intervals.
3. Save the bin edges so we can apply the same transform at deployment.

**Instructions:**  
- `N_BINS` — default number of bins.  
- `BINNING_STRATEGY` — `"quantile"` (equal-frequency), `"uniform"` (equal-width), or `"kmeans"`.  
- You can also define **custom bin edges** per column in `CUSTOM_BINS` if you need specific cut-points.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
N_BINS            = 5
BINNING_STRATEGY  = "quantile"  # "quantile", "uniform", or "kmeans"

# Optional: specify custom bin edges for specific columns.
# Example: CUSTOM_BINS = {"age": [0, 18, 35, 55, 100]}
CUSTOM_BINS = {}
# ────────────────────────────────────────────────────────────────────

bin_edges_store = {}  # will be saved for deployment

df, bin_edges_store = bin_columns(df, num_cols, N_BINS, BINNING_STRATEGY, CUSTOM_BINS, bin_edges_store)
print(f"\nOriginal numerical columns dropped. Shape: {df.shape}")

---
## 7. Categorical Columns — One-Hot Encoding

Each categorical column (including the newly binned numerical columns) will be one-hot encoded.  
The encoder is saved so the same mapping can be reused at deployment.

**Instructions:**  
- `DROP_FIRST` — set to `True` to drop one dummy per feature (avoids multicollinearity for linear models; for tree models you can leave `False`).  
- `MAX_CATEGORIES` — categories with fewer than this many occurrences are grouped into `"__rare__"`.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
DROP_FIRST      = False   # drop first dummy column per feature?
MAX_CATEGORIES  = None    # set an int to group rare categories, or None to skip
# ────────────────────────────────────────────────────────────────────

rare_mappings = {}
df, encode_cols, ohe, rare_mappings = ohe_columns(
    df, TARGET_COL, DROP_FIRST, MAX_CATEGORIES, rare_mappings
)

# Track which features came from the main data
main_feature_cols = [c for c in df.columns if c != TARGET_COL]

print(f"\nShape after one-hot encoding: {df.shape}")
df.head()

---
## 8. External Data — Load, Merge & Full Pipeline

External data from a different provider can sometimes improve model performance.  
This section puts the external features through the **exact same pipeline** as the main data:

1. Load & merge on a key
2. Drop / keep columns
3. Null analysis (same threshold by default — override if needed)
4. Class-wise variation analysis (same threshold — override if needed)
5. Binning & One-Hot Encoding

The processed external features are then concatenated with the main features **before** the train/test split.

**If you don't have external data, set `EXTERNAL_DATA_PATH = None` and skip to Section 9.**

### 8.1 Load & Merge External Data

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXTERNAL_DATA_PATH = "final_enriched_output.csv"
EXT_MERGE_KEY      = "loan_id"
# ────────────────────────────────────────────────────────────────────

ext_processed = None  # will hold the final processed external features

if EXTERNAL_DATA_PATH is not None and EXT_MERGE_KEY is not None:
    ext_raw = pd.read_csv(EXTERNAL_DATA_PATH)
    print(f"External data shape: {ext_raw.shape}")
    print(f"Columns: {ext_raw.columns.tolist()}")
    print(f"\nFirst rows:")
    display(ext_raw.head())

    # We need the original main data's merge key to align rows.
    # Re-read the full CSV, then split into dev and holdout using the same
    # holdout indices so we keep the external data perfectly aligned.
    main_raw_full = pd.read_csv(DATA_PATH)
    main_raw_full.rename(columns={"target_dpd0": TARGET_COL}, inplace=True)
    if EXT_MERGE_KEY not in main_raw_full.columns:
        raise ValueError(f"Merge key '{EXT_MERGE_KEY}' not found in main data.")
    if EXT_MERGE_KEY not in ext_raw.columns:
        raise ValueError(f"Merge key '{EXT_MERGE_KEY}' not found in external data.")

    # Reproduce the exact same holdout split to get dev vs holdout indices
    main_dev_raw, main_holdout_raw = train_test_split(
        main_raw_full,
        test_size=HOLDOUT_PCT,
        random_state=HOLDOUT_SEED,
        stratify=main_raw_full[TARGET_COL],
    )

    # ── DEV external: merge only dev rows ──────────────────────────
    main_dev_raw = main_dev_raw.reset_index(drop=True)
    ext_merged = main_dev_raw[[EXT_MERGE_KEY, TARGET_COL]].merge(
        ext_raw, on=EXT_MERGE_KEY, how="left"
    )
    ext_merged.index = df.index  # align with dev df
    ext_feature_names = [c for c in ext_merged.columns if c not in [EXT_MERGE_KEY, TARGET_COL]]

    print(f"\nMerged (dev). External feature columns ({len(ext_feature_names)}): {ext_feature_names}")
    print(f"Rows matched: {ext_merged[ext_feature_names[0]].notna().sum()} / {len(ext_merged)}")

    # Build a working df for the external pipeline (features + target for variation analysis)
    ext_df = ext_merged[ext_feature_names + [TARGET_COL]].copy()

    # ── HOLDOUT external: save raw for later scoring ───────────────
    holdout_ext_merged = main_holdout_raw[[EXT_MERGE_KEY, TARGET_COL]].merge(
        ext_raw, on=EXT_MERGE_KEY, how="left"
    )
    holdout_ext_raw = holdout_ext_merged.drop(columns=[TARGET_COL]).reset_index(drop=True)
    print(f"\nHoldout external rows saved: {len(holdout_ext_raw)}")
else:
    ext_df = None
    holdout_ext_raw = None
    print("No external data provided — skip to Section 9.")

### 8.2 External Data — Drop / Keep Columns

**Instructions:**  
Review the external columns above. Drop IDs or useless ones, or keep only selected ones.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXT_MODE = "drop"   # "drop" or "keep"

EXT_COLUMNS_LIST = [
    # "ext_id",
]  # <-- columns to drop or keep from external data
# ────────────────────────────────────────────────────────────────────

if ext_df is not None:
    if EXT_MODE == "drop":
        ext_df.drop(columns=[c for c in EXT_COLUMNS_LIST if c in ext_df.columns], inplace=True)
        print(f"Dropped from external: {EXT_COLUMNS_LIST}")
    elif EXT_MODE == "keep":
        keep = list(set(EXT_COLUMNS_LIST + [TARGET_COL]))
        ext_df = ext_df[[c for c in keep if c in ext_df.columns]]
        print(f"Kept only from external: {keep}")
    print(f"External shape after drop/keep: {ext_df.shape}")
else:
    print("No external data — skipping.")

### 8.3 External Data — Null Analysis

Same null threshold as main data by default.  
**Override** `EXT_NULL_THRESHOLD_PCT` below if you want a different threshold for external columns.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXT_NULL_THRESHOLD_PCT = NULL_THRESHOLD_PCT  # same as main — change if needed
# ────────────────────────────────────────────────────────────────────

if ext_df is not None:
    ext_feature_cols = [c for c in ext_df.columns if c != TARGET_COL]
    ext_null_drops = null_analysis(ext_df, ext_feature_cols, EXT_NULL_THRESHOLD_PCT, TARGET_COL,
                                   title="(External Data)")
    ext_df.drop(columns=ext_null_drops, inplace=True)
    print(f"\nDropped {len(ext_null_drops)} external columns above {EXT_NULL_THRESHOLD_PCT}% nulls: {ext_null_drops}")
    print(f"External shape: {ext_df.shape}")
else:
    print("No external data — skipping.")

### 8.4 External Data — Class-wise Variation

Same variation threshold as main data by default.  
**Override** `EXT_VARIATION_THRESHOLD` below if you want a different threshold.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXT_VARIATION_THRESHOLD = VARIATION_THRESHOLD  # same as main — change if needed
# ────────────────────────────────────────────────────────────────────

if ext_df is not None:
    ext_feature_cols = [c for c in ext_df.columns if c != TARGET_COL]
    ext_flagged, ext_var_df = variation_analysis(
        ext_df, ext_feature_cols, TARGET_COL, EXT_VARIATION_THRESHOLD, title="(External Data)"
    )
    print(f"\nFlagged external columns (score < {EXT_VARIATION_THRESHOLD}): {ext_flagged}")
else:
    print("No external data — skipping.")

**Instructions:**  
Review the flagged external columns. Add any you want to drop below.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXT_DROP_LOW_VARIATION = [
    # "ext_col_a",
]  # <-- paste external flagged column names to remove
# ────────────────────────────────────────────────────────────────────

if ext_df is not None:
    ext_df.drop(columns=[c for c in EXT_DROP_LOW_VARIATION if c in ext_df.columns], inplace=True)
    print(f"Dropped {len(EXT_DROP_LOW_VARIATION)} low-variation external columns.")
    print(f"External shape: {ext_df.shape}")
else:
    print("No external data — skipping.")

### 8.5 External Data — Binning & One-Hot Encoding

The remaining external columns go through the same binning + OHE process.  
Same settings as main by default — override below if needed.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
EXT_N_BINS           = N_BINS            # same as main — change if needed
EXT_BINNING_STRATEGY = BINNING_STRATEGY  # same as main — change if needed
EXT_CUSTOM_BINS      = {}                # custom bin edges for external columns
EXT_DROP_FIRST       = DROP_FIRST
EXT_MAX_CATEGORIES   = MAX_CATEGORIES
# ────────────────────────────────────────────────────────────────────

ext_bin_edges_store = {}
ext_rare_mappings   = {}
ext_ohe             = None
ext_encode_cols     = []
ext_cat_cols        = []
ext_num_cols        = []

if ext_df is not None:
    ext_feature_cols = [c for c in ext_df.columns if c != TARGET_COL]

    if len(ext_feature_cols) == 0:
        print("No external feature columns survived preprocessing. Skipping.")
        ext_df = None
    else:
        # Identify types
        ext_cat_cols, ext_num_cols = identify_col_types(ext_df, ext_feature_cols)
        print(f"External categorical ({len(ext_cat_cols)}): {ext_cat_cols}")
        print(f"External numerical  ({len(ext_num_cols)}): {ext_num_cols}")

        # Bin numerical
        if ext_num_cols:
            ext_df, ext_bin_edges_store = bin_columns(
                ext_df, ext_num_cols, EXT_N_BINS, EXT_BINNING_STRATEGY,
                EXT_CUSTOM_BINS, ext_bin_edges_store
            )
            print(f"\nExternal numerical columns binned and dropped. Shape: {ext_df.shape}")

        # OHE all
        ext_df, ext_encode_cols, ext_ohe, ext_rare_mappings = ohe_columns(
            ext_df, TARGET_COL, EXT_DROP_FIRST, EXT_MAX_CATEGORIES, ext_rare_mappings
        )
        ext_processed_cols = [c for c in ext_df.columns if c != TARGET_COL]
        print(f"\nExternal OHE done. {len(ext_processed_cols)} features.")
else:
    print("No external data — skipping.")

### 8.6 Concatenate External Features with Main Data

In [ ]:
if ext_df is not None:
    ext_only = ext_df.drop(columns=[TARGET_COL])  # don't duplicate target
    before_cols = df.shape[1]
    df = pd.concat([df.drop(columns=[TARGET_COL]), ext_only, df[[TARGET_COL]]], axis=1)
    after_cols = df.shape[1]
    print(f"Main features: {before_cols - 1}  +  External features: {ext_only.shape[1]}  =  Total: {after_cols - 1}")
    print(f"Combined shape: {df.shape}")
else:
    print(f"No external data added. Shape: {df.shape}")

---
## 9. Save All Transformation Artefacts

Everything needed to reproduce these transformations on new data is saved here:  
- Bin edges — main + external (JSON)  
- OneHotEncoders — main + external (joblib)  
- Rare-category mappings (JSON)  
- Column metadata (JSON)  

At deployment, load these artefacts and apply the same pipeline.

In [ ]:
# ── Main artefacts ─────────────────────────────────────────────────
with open(os.path.join(ARTEFACT_DIR, "bin_edges.json"), "w") as f:
    json.dump(bin_edges_store, f, indent=2)

joblib.dump(ohe, os.path.join(ARTEFACT_DIR, "ohe_encoder.joblib"))

with open(os.path.join(ARTEFACT_DIR, "rare_mappings.json"), "w") as f:
    json.dump(rare_mappings, f, indent=2)

# ── External artefacts (if any) ────────────────────────────────────
if ext_df is not None:
    with open(os.path.join(ARTEFACT_DIR, "ext_bin_edges.json"), "w") as f:
        json.dump(ext_bin_edges_store, f, indent=2)

    joblib.dump(ext_ohe, os.path.join(ARTEFACT_DIR, "ext_ohe_encoder.joblib"))

    with open(os.path.join(ARTEFACT_DIR, "ext_rare_mappings.json"), "w") as f:
        json.dump(ext_rare_mappings, f, indent=2)

# ── Column metadata ────────────────────────────────────────────────
col_meta = {
    "original_cat_cols": cat_cols,
    "original_num_cols": num_cols,
    "encode_cols": encode_cols,
    "main_feature_cols": main_feature_cols,
    "ext_cat_cols": ext_cat_cols,
    "ext_num_cols": ext_num_cols,
    "ext_encode_cols": ext_encode_cols,
    "final_feature_cols": [c for c in df.columns if c != TARGET_COL],
    "target_col": TARGET_COL,
    "has_external": ext_df is not None,
    "ext_merge_key": EXT_MERGE_KEY if ext_df is not None else None,
}
with open(os.path.join(ARTEFACT_DIR, "column_metadata.json"), "w") as f:
    json.dump(col_meta, f, indent=2)

# ── Holdout metadata ──────────────────────────────────────────────
holdout_meta = {
    "holdout_pct": HOLDOUT_PCT,
    "holdout_seed": HOLDOUT_SEED,
    "holdout_rows": len(holdout_raw),
    "dev_rows": len(df),
}
with open(os.path.join(ARTEFACT_DIR, "holdout_meta.json"), "w") as f:
    json.dump(holdout_meta, f, indent=2)

print("All transformation artefacts saved to:", ARTEFACT_DIR)
print("Files:", os.listdir(ARTEFACT_DIR))

---
## 10. Stratified Train / Test Split

We use **stratified splitting** so both train and test sets preserve the original class ratio.

**Instructions:**  
- `TEST_SIZE` — fraction of data reserved for testing.  
- `RANDOM_STATE` — seed for reproducibility.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
TEST_SIZE    = 0.2
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────────────────

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train target dist:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest target dist:\n{y_test.value_counts(normalize=True)}")

---
## 11. XGBoost — GridSearchCV for Hyperparameters

We use `GridSearchCV` with **stratified k-fold** cross-validation to find the best XGBoost parameters.

**Instructions:**  
- Edit `param_grid` to change the search space.  
- Set `CV_FOLDS` and `SCORING` metric.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
CV_FOLDS = 5
SCORING  = "roc_auc"  # scoring metric for grid search

param_grid = {
    "n_estimators":     [100, 200, 300],
    "max_depth":        [3, 5, 7],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}
# ────────────────────────────────────────────────────────────────────

# Calculate scale_pos_weight for imbalanced data
neg, pos = np.bincount(y_train.astype(int))
scale_pos_weight = neg / pos if pos > 0 else 1
print(f"scale_pos_weight = {scale_pos_weight:.2f} (neg={neg}, pos={pos})")

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    verbosity=0,
)

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring=SCORING,
    cv=skf,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

print("\nStarting GridSearchCV ... (this may take a while)")
grid_search.fit(X_train, y_train)

print(f"\nBest {SCORING}: {grid_search.best_score_:.4f}")
print(f"Best params: {grid_search.best_params_}")

In [ ]:
# Store the best estimator
best_model = grid_search.best_estimator_

# Show top-10 parameter combos
cv_results = pd.DataFrame(grid_search.cv_results_)
top10 = cv_results.nsmallest(10, "rank_test_score")[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
]
top10

---
## 12. Feature Importance — Smart Auto-Flag & Selection

We compute two importance measures:
1. **XGBoost gain** — how much each feature contributes to splits (can be inflated by high-cardinality noise).
2. **Permutation importance** — actual AUC drop when a feature is shuffled (true predictive value).

Features that are **suspicious** are automatically flagged:
- High XGBoost gain rank (top percentile) **BUT** low permutation importance → likely high-cardinality noise that the tree memorises but doesn't genuinely help.

These features are **auto-dropped**, but you can **add any back** in the next cell.

**Instructions:**  
- `GAIN_TOP_PCT` — percentile threshold for "high gain" (default 50 = top half).  
- `PERM_MIN_THRESHOLD` — permutation importance below this is considered negligible.

In [ ]:
from sklearn.inspection import permutation_importance

# ── USER INPUT ──────────────────────────────────────────────────────
GAIN_TOP_PCT        = 50     # features in the top X% by XGBoost gain
PERM_MIN_THRESHOLD  = 0.001  # permutation importance below this = negligible
# ────────────────────────────────────────────────────────────────────

# 1. XGBoost gain importance
imp = pd.Series(best_model.feature_importances_, index=X_train.columns)
imp_sorted = imp.sort_values(ascending=False)

# 2. Permutation importance
perm_result = permutation_importance(
    best_model, X_test, y_test, n_repeats=10,
    random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1,
)
perm_imp = pd.Series(perm_result.importances_mean, index=X_test.columns)
perm_sorted = perm_imp.sort_values(ascending=False)

# ── Visualise ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, min(30, len(imp_sorted)) * 0.25)))

top_n = min(30, len(imp_sorted))
sns.barplot(x=imp_sorted.values[:top_n], y=imp_sorted.index[:top_n],
            palette="viridis", ax=axes[0])
axes[0].set_title("XGBoost Feature Importance (Gain)")
axes[0].set_xlabel("Importance")

sns.barplot(x=perm_sorted.values[:top_n], y=perm_sorted.index[:top_n],
            palette="magma", ax=axes[1])
axes[1].set_title("Permutation Importance (AUC drop)")
axes[1].set_xlabel("Mean AUC decrease")

plt.tight_layout()
plt.show()

# ── Build summary table ────────────────────────────────────────────
summary = pd.DataFrame({
    "xgb_gain": imp,
    "xgb_gain_rank": imp.rank(ascending=False).astype(int),
    "perm_importance": perm_imp,
}).sort_values("xgb_gain", ascending=False)

# ── Flag suspicious features ───────────────────────────────────────
gain_cutoff = np.percentile(imp.values, 100 - GAIN_TOP_PCT)
summary["high_gain"]  = summary["xgb_gain"] >= gain_cutoff
summary["low_perm"]   = summary["perm_importance"] < PERM_MIN_THRESHOLD
summary["suspicious"] = summary["high_gain"] & summary["low_perm"]

suspicious_features = summary[summary["suspicious"]].index.tolist()

print(f"\nGain cutoff (top {GAIN_TOP_PCT}%): {gain_cutoff:.6f}")
print(f"Perm importance threshold: {PERM_MIN_THRESHOLD}")
print(f"\n{'='*70}")
print(f"SUSPICIOUS FEATURES (high XGBoost gain + negligible AUC contribution):")
print(f"{'='*70}")
if suspicious_features:
    display(summary.loc[suspicious_features, ["xgb_gain", "xgb_gain_rank", "perm_importance"]])
else:
    print("None found — all high-gain features also have meaningful permutation importance.")

print(f"\nFull summary table:")
summary

### 12.1 Auto-Drop Suspicious Features & Add Back

The suspicious features listed above will be **dropped automatically**.  
If you disagree with any removal, **add them back** in `ADD_BACK_FEATURES` below.

In [ ]:
# ── USER INPUT ──────────────────────────────────────────────────────
# Features that were auto-flagged but you want to KEEP:
ADD_BACK_FEATURES = [
    # "feature_that_was_flagged_but_I_want_to_keep",
]

# Additional features you want to drop manually (beyond auto-flagged):
EXTRA_DROP_FEATURES = [
    # "some_other_feature",
]
# ────────────────────────────────────────────────────────────────────

# Compute final drop list
auto_drop = [f for f in suspicious_features if f not in ADD_BACK_FEATURES]
final_drop = list(set(auto_drop + EXTRA_DROP_FEATURES))

if final_drop:
    print(f"Dropping {len(final_drop)} features:")
    for f in final_drop:
        gain_val = summary.loc[f, 'xgb_gain'] if f in summary.index else 'N/A'
        perm_val = summary.loc[f, 'perm_importance'] if f in summary.index else 'N/A'
        source = "auto-flagged" if f in auto_drop else "manual"
        print(f"  - {f}  (gain={gain_val}, perm={perm_val}, {source})")

    X_train = X_train.drop(columns=final_drop, errors="ignore")
    X_test  = X_test.drop(columns=final_drop, errors="ignore")

    print(f"\nRetraining model on {X_train.shape[1]} features ...")
    best_model.fit(X_train, y_train)
    print("Retrained.")
else:
    print("No features dropped — all features look clean.")

if ADD_BACK_FEATURES:
    print(f"\nKept (added back): {ADD_BACK_FEATURES}")

print(f"Final feature count: {X_train.shape[1]}")

---
## 13. Evaluation Metrics

We compute and visualise the following metrics on the **test set**:

| Metric | What it tells you |
|--------|-------------------|
| **Accuracy** | Overall correctness |
| **Balanced Accuracy** | Average recall per class — good for imbalanced data |
| **F1 Score** | Harmonic mean of precision and recall |
| **AUC (ROC)** | Rank-ordering ability |
| **Gini** | `2 x AUC - 1` — common in credit scoring |

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

acc      = accuracy_score(y_test, y_pred)
bal_acc  = balanced_accuracy_score(y_test, y_pred)
f1       = f1_score(y_test, y_pred)
auc      = roc_auc_score(y_test, y_prob)
gini     = 2 * auc - 1

metrics = {
    "Accuracy":          acc,
    "Balanced Accuracy": bal_acc,
    "F1 Score":          f1,
    "AUC (ROC)":         auc,
    "Gini":              gini,
}

print("="*50)
print("         TEST SET METRICS")
print("="*50)
for name, val in metrics.items():
    bar = "|" + "#" * int(val * 40) + " " * (40 - int(val * 40)) + "|"
    print(f"  {name:<20s}  {val:.4f}  {bar}")
print("="*50)
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# ── Beautiful metric visualisations ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# ── 1. ROC Curve ───────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0, 0].plot(fpr, tpr, lw=2, color="#1f77b4",
                label=f"AUC = {auc:.4f}")
axes[0, 0].plot([0, 1], [0, 1], "--", color="grey")
axes[0, 0].fill_between(fpr, tpr, alpha=0.15, color="#1f77b4")
axes[0, 0].set_xlabel("False Positive Rate")
axes[0, 0].set_ylabel("True Positive Rate")
axes[0, 0].set_title("ROC Curve")
axes[0, 0].legend(loc="lower right", fontsize=12)

# ── 2. Precision-Recall Curve ──────────────────────────────────────
prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[0, 1].plot(rec, prec, lw=2, color="#ff7f0e")
axes[0, 1].fill_between(rec, prec, alpha=0.15, color="#ff7f0e")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].set_title("Precision-Recall Curve")

# ── 3. Confusion Matrix ────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1, 0],
            xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
axes[1, 0].set_xlabel("Predicted")
axes[1, 0].set_ylabel("Actual")
axes[1, 0].set_title("Confusion Matrix")

# ── 4. Metrics Bar Chart ───────────────────────────────────────────
colors_bar = ["#2ca02c", "#17becf", "#ff7f0e", "#1f77b4", "#9467bd"]
bars = axes[1, 1].bar(metrics.keys(), metrics.values(), color=colors_bar,
                       edgecolor="black", linewidth=0.8)
for bar, val in zip(bars, metrics.values()):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.4f}", ha="center", fontsize=11, fontweight="bold")
axes[1, 1].set_ylim(0, 1.1)
axes[1, 1].set_title("Metric Summary")
axes[1, 1].tick_params(axis="x", rotation=25)

plt.suptitle("Model Evaluation Dashboard", fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 14. Save Final Model & Artefacts for Deployment

We save:
- The trained XGBoost model
- The final feature list (after suspicious feature removal)
- A summary of metrics
- Best hyperparameters

Together with the transformation artefacts saved in Section 9, this gives you everything needed to deploy.

In [ ]:
# Save model
joblib.dump(best_model, os.path.join(ARTEFACT_DIR, "xgb_model.joblib"))

# Save final feature list
with open(os.path.join(ARTEFACT_DIR, "final_features.json"), "w") as f:
    json.dump(X_train.columns.tolist(), f, indent=2)

# Save metrics
with open(os.path.join(ARTEFACT_DIR, "test_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Save best hyperparameters
with open(os.path.join(ARTEFACT_DIR, "best_params.json"), "w") as f:
    json.dump(grid_search.best_params_, f, indent=2)

# Save dropped feature info
with open(os.path.join(ARTEFACT_DIR, "dropped_features.json"), "w") as f:
    json.dump({
        "auto_flagged_suspicious": suspicious_features,
        "added_back": ADD_BACK_FEATURES,
        "extra_manual_drops": EXTRA_DROP_FEATURES,
        "final_dropped": final_drop if final_drop else [],
    }, f, indent=2)

print("All deployment artefacts saved.")
print("Contents of artefact directory:")
for fname in sorted(os.listdir(ARTEFACT_DIR)):
    size = os.path.getsize(os.path.join(ARTEFACT_DIR, fname))
    print(f"  {fname:<35s}  {size:>8,} bytes")

---
## 15. How to Use These Artefacts at Deployment

Below is a reference function that loads all artefacts and scores a new raw dataframe.  
It handles both main and external data transformations.

```python
import joblib, json, pandas as pd, numpy as np

def score_new_data(raw_df, artefact_dir, ext_raw_df=None, merge_key=None):
    # 1. Load artefacts
    model     = joblib.load(f"{artefact_dir}/xgb_model.joblib")
    ohe       = joblib.load(f"{artefact_dir}/ohe_encoder.joblib")
    bin_edges = json.load(open(f"{artefact_dir}/bin_edges.json"))
    col_meta  = json.load(open(f"{artefact_dir}/column_metadata.json"))
    rare_map  = json.load(open(f"{artefact_dir}/rare_mappings.json"))
    features  = json.load(open(f"{artefact_dir}/final_features.json"))

    # 2. Bin numerical columns
    for col, info in bin_edges.items():
        edges = info["edges"]
        raw_df[col + "_bin"] = pd.cut(raw_df[col], bins=edges, labels=False, include_lowest=True)
    raw_df.drop(columns=col_meta["original_num_cols"], inplace=True, errors="ignore")

    # 3. Handle rare categories
    for col, rares in rare_map.items():
        if col in raw_df.columns:
            raw_df[col] = raw_df[col].apply(lambda x: "__rare__" if x in rares else x)

    # 4. Fill nulls & encode main features
    encode_cols = col_meta["encode_cols"]
    for col in encode_cols:
        if col in raw_df.columns and raw_df[col].isnull().any():
            raw_df[col] = raw_df[col].fillna("__NULL__")
    raw_df[encode_cols] = raw_df[encode_cols].astype(str)
    encoded = ohe.transform(raw_df[encode_cols])
    enc_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(encode_cols),
                          index=raw_df.index)

    # 5. External features (if applicable)
    if col_meta.get("has_external") and ext_raw_df is not None:
        ext_ohe       = joblib.load(f"{artefact_dir}/ext_ohe_encoder.joblib")
        ext_bin_edges = json.load(open(f"{artefact_dir}/ext_bin_edges.json"))
        ext_rare_map  = json.load(open(f"{artefact_dir}/ext_rare_mappings.json"))

        if merge_key:
            ext_raw_df = raw_df[[merge_key]].merge(ext_raw_df, on=merge_key, how="left")

        for col, info in ext_bin_edges.items():
            ext_raw_df[col + "_bin"] = pd.cut(ext_raw_df[col], bins=info["edges"],
                                              labels=False, include_lowest=True)
        ext_raw_df.drop(columns=col_meta["ext_num_cols"], inplace=True, errors="ignore")

        for col, rares in ext_rare_map.items():
            if col in ext_raw_df.columns:
                ext_raw_df[col] = ext_raw_df[col].apply(
                    lambda x: "__rare__" if x in rares else x)

        ext_enc_cols = col_meta["ext_encode_cols"]
        for col in ext_enc_cols:
            if col in ext_raw_df.columns and ext_raw_df[col].isnull().any():
                ext_raw_df[col] = ext_raw_df[col].fillna("__NULL__")
        ext_raw_df[ext_enc_cols] = ext_raw_df[ext_enc_cols].astype(str)
        ext_encoded = ext_ohe.transform(ext_raw_df[ext_enc_cols])
        ext_enc_df = pd.DataFrame(ext_encoded,
                                  columns=ext_ohe.get_feature_names_out(ext_enc_cols),
                                  index=raw_df.index)
        enc_df = pd.concat([enc_df, ext_enc_df], axis=1)

    # 6. Align to training features and predict
    for col in features:
        if col not in enc_df.columns:
            enc_df[col] = 0
    enc_df = enc_df[features]

    return model.predict_proba(enc_df)[:, 1]
```

---
## 16. Holdout Evaluation — True Out-of-Sample Test

This section scores the **raw 15% holdout** data through the deployment pipeline (`score_new_data()`).  
The holdout was set aside **before any preprocessing** — it has never been seen by any encoder, binner, or the model.

This is the closest simulation of **production scoring** and validates:
1. The model generalises beyond the dev set
2. The saved artefacts / deployment pipeline work correctly
3. There is no data leakage inflating metrics

In [ ]:
# ── Score the raw holdout through the deployment pipeline ──────────
# This uses the same score_new_data() logic with saved artefacts,
# exactly as production would.

def score_new_data(raw_df, artefact_dir, ext_raw_df=None, merge_key=None):
    """Score raw data using saved artefacts. Identical to deployment."""
    model     = joblib.load(os.path.join(artefact_dir, "xgb_model.joblib"))
    ohe_enc   = joblib.load(os.path.join(artefact_dir, "ohe_encoder.joblib"))
    bin_edges = json.load(open(os.path.join(artefact_dir, "bin_edges.json")))
    col_meta  = json.load(open(os.path.join(artefact_dir, "column_metadata.json")))
    rare_map  = json.load(open(os.path.join(artefact_dir, "rare_mappings.json")))
    features  = json.load(open(os.path.join(artefact_dir, "final_features.json")))

    raw_df = raw_df.copy()

    # Bin numerical columns
    for col, info in bin_edges.items():
        edges = info["edges"]
        if col in raw_df.columns:
            raw_df[col + "_bin"] = pd.cut(raw_df[col], bins=edges, labels=False, include_lowest=True)
    raw_df.drop(columns=[c for c in col_meta["original_num_cols"] if c in raw_df.columns],
                inplace=True, errors="ignore")

    # Handle rare categories
    for col, rares in rare_map.items():
        if col in raw_df.columns:
            raw_df[col] = raw_df[col].apply(lambda x: "__rare__" if x in rares else x)

    # Fill nulls & OHE main features
    enc_cols = col_meta["encode_cols"]
    for col in enc_cols:
        if col in raw_df.columns and raw_df[col].isnull().any():
            raw_df[col] = raw_df[col].fillna("__NULL__")
    for col in enc_cols:
        if col not in raw_df.columns:
            raw_df[col] = "__NULL__"
    raw_df[enc_cols] = raw_df[enc_cols].astype(str)
    encoded = ohe_enc.transform(raw_df[enc_cols])
    enc_df = pd.DataFrame(encoded, columns=ohe_enc.get_feature_names_out(enc_cols),
                          index=raw_df.index)

    # External features (if applicable)
    if col_meta.get("has_external") and ext_raw_df is not None:
        ext_raw_df = ext_raw_df.copy()
        ext_ohe_enc   = joblib.load(os.path.join(artefact_dir, "ext_ohe_encoder.joblib"))
        ext_bin_edges = json.load(open(os.path.join(artefact_dir, "ext_bin_edges.json")))
        ext_rare_map  = json.load(open(os.path.join(artefact_dir, "ext_rare_mappings.json")))

        if merge_key and merge_key in raw_df.columns:
            ext_raw_df = raw_df[[merge_key]].merge(ext_raw_df, on=merge_key, how="left")
            ext_raw_df.index = raw_df.index

        for col, info in ext_bin_edges.items():
            if col in ext_raw_df.columns:
                ext_raw_df[col + "_bin"] = pd.cut(ext_raw_df[col], bins=info["edges"],
                                                  labels=False, include_lowest=True)
        ext_raw_df.drop(columns=[c for c in col_meta["ext_num_cols"] if c in ext_raw_df.columns],
                        inplace=True, errors="ignore")

        for col, rares in ext_rare_map.items():
            if col in ext_raw_df.columns:
                ext_raw_df[col] = ext_raw_df[col].apply(
                    lambda x: "__rare__" if x in rares else x)

        ext_enc_cols = col_meta["ext_encode_cols"]
        for col in ext_enc_cols:
            if col in ext_raw_df.columns and ext_raw_df[col].isnull().any():
                ext_raw_df[col] = ext_raw_df[col].fillna("__NULL__")
        for col in ext_enc_cols:
            if col not in ext_raw_df.columns:
                ext_raw_df[col] = "__NULL__"
        ext_raw_df[ext_enc_cols] = ext_raw_df[ext_enc_cols].astype(str)
        ext_encoded = ext_ohe_enc.transform(ext_raw_df[ext_enc_cols])
        ext_enc_df = pd.DataFrame(ext_encoded,
                                  columns=ext_ohe_enc.get_feature_names_out(ext_enc_cols),
                                  index=raw_df.index)
        enc_df = pd.concat([enc_df, ext_enc_df], axis=1)

    # Align to training features and predict
    for col in features:
        if col not in enc_df.columns:
            enc_df[col] = 0
    enc_df = enc_df[features]

    return model.predict_proba(enc_df)[:, 1]


# ── Prepare holdout data ──────────────────────────────────────────
# Select only the columns the pipeline expects (same COLUMNS_LIST as Section 2)
holdout_df = holdout_raw.copy()
if MODE == "keep":
    keep_cols = list(set(COLUMNS_LIST + [TARGET_COL]))
    holdout_df = holdout_df[[c for c in keep_cols if c in holdout_df.columns]]
elif MODE == "drop":
    holdout_df.drop(columns=[c for c in COLUMNS_LIST if c in holdout_df.columns], inplace=True)

y_holdout = holdout_df[TARGET_COL]
holdout_features = holdout_df.drop(columns=[TARGET_COL])

# Score through deployment pipeline
h_ext = holdout_ext_raw if (EXTERNAL_DATA_PATH is not None and holdout_ext_raw is not None) else None
h_merge = EXT_MERGE_KEY if (EXTERNAL_DATA_PATH is not None) else None

h_prob = score_new_data(
    holdout_features, ARTEFACT_DIR,
    ext_raw_df=h_ext,
    merge_key=h_merge,
)
h_pred = (h_prob >= 0.5).astype(int)

# ── Compute holdout metrics ────────────────────────────────────────
h_acc     = accuracy_score(y_holdout, h_pred)
h_bal_acc = balanced_accuracy_score(y_holdout, h_pred)
h_f1      = f1_score(y_holdout, h_pred)
h_auc     = roc_auc_score(y_holdout, h_prob)
h_gini    = 2 * h_auc - 1

holdout_metrics = {
    "Accuracy":          h_acc,
    "Balanced Accuracy": h_bal_acc,
    "F1 Score":          h_f1,
    "AUC (ROC)":         h_auc,
    "Gini":              h_gini,
}

print("=" * 60)
print("         HOLDOUT (OUT-OF-SAMPLE) METRICS")
print("=" * 60)
for name, val in holdout_metrics.items():
    bar = "|" + "#" * int(val * 40) + " " * (40 - int(val * 40)) + "|"
    print(f"  {name:<20s}  {val:.4f}  {bar}")
print("=" * 60)
print(f"\nClassification Report (Holdout):\n")
print(classification_report(y_holdout, h_pred))

In [ ]:
# ── Dev Test vs Holdout — Side-by-Side Comparison ─────────────────
compare_df = pd.DataFrame({
    "Dev Test": metrics,
    "Holdout (OOS)": holdout_metrics,
})
compare_df["Difference"] = compare_df["Holdout (OOS)"] - compare_df["Dev Test"]
compare_df["Diff %"] = (compare_df["Difference"] / compare_df["Dev Test"] * 100).round(2)

print("=" * 70)
print("         DEV TEST vs HOLDOUT COMPARISON")
print("=" * 70)
print(compare_df.to_string())
print("=" * 70)

# ── Visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Side-by-side bar chart
x = np.arange(len(metrics))
width = 0.35
bars1 = axes[0].bar(x - width/2, list(metrics.values()), width,
                     label="Dev Test", color="#1f77b4", edgecolor="black", linewidth=0.6)
bars2 = axes[0].bar(x + width/2, list(holdout_metrics.values()), width,
                     label="Holdout (OOS)", color="#ff7f0e", edgecolor="black", linewidth=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(list(metrics.keys()), rotation=25, ha="right")
axes[0].set_ylim(0, 1.15)
axes[0].legend(fontsize=11)
axes[0].set_title("Dev Test vs Holdout Metrics")
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{bar.get_height():.3f}", ha="center", fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{bar.get_height():.3f}", ha="center", fontsize=9)

# 2. ROC overlay
h_fpr, h_tpr, _ = roc_curve(y_holdout, h_prob)
d_fpr, d_tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(d_fpr, d_tpr, lw=2, color="#1f77b4", label=f"Dev Test AUC={auc:.4f}")
axes[1].plot(h_fpr, h_tpr, lw=2, color="#ff7f0e", label=f"Holdout AUC={h_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].fill_between(d_fpr, d_tpr, alpha=0.08, color="#1f77b4")
axes[1].fill_between(h_fpr, h_tpr, alpha=0.08, color="#ff7f0e")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Dev Test vs Holdout")
axes[1].legend(loc="lower right", fontsize=11)

plt.suptitle("Model Generalisation Check", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Save holdout metrics ──────────────────────────────────────────
with open(os.path.join(ARTEFACT_DIR, "holdout_metrics.json"), "w") as f:
    json.dump(holdout_metrics, f, indent=2)

print(f"\nHoldout metrics saved to {ARTEFACT_DIR}/holdout_metrics.json")

---

**You're done!**  
Go back to any section to tweak thresholds, change binning strategies, or add features — then re-run from that point onward.

**Standalone scripts** (same logic, no notebook required):
- `train_pipeline.py` — runs the full training pipeline end-to-end
- `score_new_data.py` — scores new raw data using saved artefacts